# Lab 4: Leakage-safe pipelines, cross-validation and tuning

**Problem:** using support ticket data, predict if a ticket gets escalated (A) and how many hours it takes to resolve (B).

## Setup

In [17]:
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error

# Experiment A: Classification (predict `escalated`)

## A0. Load and explore

In [18]:
clf_df = pd.read_csv("data/classification_escalation.csv")
display(clf_df.head())

# missing values per column
print(clf_df.isna().sum())

# share of escalated = 0 vs 1 (class balance)
print(clf_df["escalated"].value_counts(normalize=True))

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64
escalated
0    0.787778
1    0.212222
Name: proportion, dtype: float64


All features have some missing values, so they need filling. Only 21% of tickets are escalated, so I stratify and use F1 instead of accuracy.

## A1. Features, target and stratified train/test split

In [19]:
Xc = clf_df.drop(columns=["ticket_id", "escalated"])
yc = clf_df["escalated"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, stratify=yc, random_state=31
)

print(Xc_train.shape, Xc_test.shape)
print(yc_train.value_counts(normalize=True))
print(yc_test.value_counts(normalize=True))

(720, 5) (180, 5)
escalated
0    0.7875
1    0.2125
Name: proportion, dtype: float64
escalated
0    0.788889
1    0.211111
Name: proportion, dtype: float64


I dropped `ticket_id` since it's just an ID. 80/20 split, stratified so both parts keep about 21% escalated.

## A2. Preprocessing

In [20]:
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

# quick look at the output shape
print(preprocess.fit_transform(Xc_train).shape)

(720, 11)


Numbers: median fill and scaling (so big columns like tenure don't dominate). Categories: most frequent fill and one-hot. It's all inside the pipeline so it's only fitted on training data (no leakage).

## A3. Comparing candidate models with 5-fold stratified cross-validation

In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "k-NN": KNeighborsClassifier(),
    "Decision tree": DecisionTreeClassifier(random_state=42),
}

rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_validate(pipe, Xc_train, yc_train, cv=cv, scoring="f1")
    f1s = scores["test_score"]          # the 5 fold scores
    rows.append({"model": name,
                 "F1 mean": f1s.mean(),
                 "F1 std": f1s.std(),
                 "fold scores": np.round(f1s, 3)})

cv_results_clf = pd.DataFrame(rows)
cv_results_clf

,model,F1 mean,F1 std,fold scores
0,Logistic regression,0.430311,0.051932,"[0.341, 0.449, 0.417, 0.444, 0.5]"
1,k-NN,0.321515,0.046374,"[0.372, 0.238, 0.311, 0.333, 0.353]"
2,Decision tree,0.374999,0.020537,"[0.355, 0.412, 0.358, 0.381, 0.369]"


5-fold stratified CV on the training set, same folds for every model, scored with F1. I chose 3 different kinds of model: linear (logistic regression), similarity based (k-NN) and rule based (tree).

Logistic regression is best, so I tune it next.

## A4. Hyperparameter tuning (logistic regression)

In [22]:
lr_pipe = Pipeline([("prep", preprocess),
                    ("model", LogisticRegression(max_iter=1000))])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
}

grid = GridSearchCV(lr_pipe, param_grid, cv=cv, scoring="f1")
grid.fit(Xc_train, yc_train)

print("Best settings:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 3))

# full table of all 10 combinations
res = pd.DataFrame(grid.cv_results_)
res[["param_model__C", "param_model__class_weight",
     "mean_test_score", "std_test_score", "rank_test_score"]].round(3)

Best settings: {'model__C': 0.1, 'model__class_weight': 'balanced'}
Best CV F1: 0.559


,param_model__C,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
0,0.01,None,0.000,0.000,10
1,0.01,balanced,0.540,0.024,5
2,0.10,None,0.373,0.055,9
3,0.10,balanced,0.559,0.034,1
4,1.00,None,0.430,0.052,6
5,1.00,balanced,0.548,0.036,4
6,10.00,None,0.429,0.054,7
7,10.00,balanced,0.548,0.037,2
8,100.00,None,0.429,0.054,7
9,100.00,balanced,0.548,0.037,2


I tuned logistic regression because it scored best. `C` is the penalty strength and `class_weight="balanced"` helps with the imbalance. Best: C = 0.1, balanced, CV F1 = 0.559 (up from 0.430). Most of the gain came from `class_weight`.

## A5: Final test evaluation

In [23]:
best_clf = grid.best_estimator_
yc_pred = best_clf.predict(Xc_test)

print(confusion_matrix(yc_test, yc_pred))
print(classification_report(yc_test, yc_pred, digits=3))


[[101  41]
 [ 12  26]]
              precision    recall  f1-score   support

           0      0.894     0.711     0.792       142
           1      0.388     0.684     0.495        38

    accuracy                          0.706       180
   macro avg      0.641     0.698     0.644       180
weighted avg      0.787     0.706     0.729       180



Test F1 = 0.495 (CV was 0.559, so a small drop, which is normal). The model catches 26 of 38 escalations (recall 0.68) but raises 41 false alarms (precision 0.39). I think missing an escalation (false negative) is worse than a false alarm, so this trade-off is acceptable.

# Experiment B: Regression (predict resolution_time_hours)


## B0. Load and explore

In [24]:
reg_df = pd.read_csv("data/resolution_time_regression.csv")
display(reg_df.head())

# missing values per column
print(reg_df.isna().sum())

# summary of the target
print(reg_df["resolution_time_hours"].describe())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64
count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64


## B1. Split

In [26]:
Xr = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
yr = reg_df["resolution_time_hours"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=31
)
print(Xr_train.shape, Xr_test.shape)

(640, 5) (160, 5)


## B2. Comparing linear regression, Ridge and a tree

In [27]:
cv_r = KFold(n_splits=5, shuffle=True, random_state=42)

reg_models = {
    "Linear regression": LinearRegression(),
    "Ridge": Ridge(),
    "Decision tree": DecisionTreeRegressor(random_state=42),
}

rows = []
for name, model in reg_models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    s = cross_validate(pipe, Xr_train, yr_train, cv=cv_r,
                       scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"])
    mae = -s["test_neg_mean_absolute_error"]
    rmse = -s["test_neg_root_mean_squared_error"]
    rows.append({"model": name, "MAE mean": mae.mean(), "MAE std": mae.std(),
                 "RMSE mean": rmse.mean()})

pd.DataFrame(rows).round(3)

,model,MAE mean,MAE std,RMSE mean
0,Linear regression,2.041,0.065,3.015
1,Ridge,2.043,0.066,3.014
2,Decision tree,3.097,0.086,4.424


Plain split and KFold here, since the target is a number and can't be stratified. I use MAE because it's the average error in hours, easy to understand. Linear and Ridge are almost the same (MAE about 2.04 h). The tree is much worse (3.10 h) because it has no depth limit and overfits.

## B3. Tuning Ridge alpha

In [28]:
ridge_pipe = Pipeline([("prep", preprocess), ("model", Ridge())])

ridge_grid = GridSearchCV(ridge_pipe,
                          {"model__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                          cv=cv_r, scoring="neg_mean_absolute_error")
ridge_grid.fit(Xr_train, yr_train)

print("Best alpha:", ridge_grid.best_params_)
print("Best CV MAE:", round(-ridge_grid.best_score_, 3))

r = pd.DataFrame(ridge_grid.cv_results_)
r["MAE mean"] = -r["mean_test_score"]
r[["param_model__alpha", "MAE mean", "std_test_score"]].round(3)

Best alpha: {'model__alpha': 0.01}
Best CV MAE: 2.041


,param_model__alpha,MAE mean,std_test_score
0,0.01,2.041,0.065
1,0.10,2.041,0.065
2,1.00,2.043,0.066
3,10.00,2.056,0.076
4,100.00,2.212,0.150
5,1000.00,2.762,0.232


I tuned Ridge alpha from 0.01 to 1000. Best was alpha = 0.01 (CV MAE 2.041 h, std 0.065), but anything up to 10 scores about the same, so the penalty doesn't really help. There are only 11 features and 640 tickets, so the model isn't overfitting. Large alpha (100, 1000) makes MAE worse because the model becomes too simple.

## B4. Final test, once

In [29]:
best_reg = ridge_grid.best_estimator_
yr_pred = best_reg.predict(Xr_test)

mae = mean_absolute_error(yr_test, yr_pred)
rmse = np.sqrt(np.mean((yr_test - yr_pred) ** 2))
print("Test MAE:", round(mae, 3), "  Test RMSE:", round(rmse, 3))

# the 5 biggest misses (in hours)
errors = (yr_test - yr_pred).abs().sort_values(ascending=False)
print(errors.head().round(1))

# how many very long tickets are there in the data?
print("Tickets over 15 h:", (yr > 15).sum())

Test MAE: 2.277   Test RMSE: 3.331
188    19.7
19     14.2
502    14.1
113     6.8
505     6.1
Name: resolution_time_hours, dtype: float64
Tickets over 15 h: 15


Final model: Ridge (alpha = 0.01). Test MAE = 2.28 h, RMSE = 3.33 h. A few very long tickets (up to 27.5 h) get missed by 14 to 20 h. These outliers push RMSE up because it squares errors, while MAE treats every hour the same. So RMSE can prefer a model that avoids big misses, and MAE can prefer one that's better on normal tickets.

## Conclusion

**Final models.** A: logistic regression with C = 0.1 and balanced class weights. It had the best CV F1 (0.559), is simple and fast, and its weights can be explained. B: Ridge with alpha = 0.01. Linear models beat the tree by about 1 hour of MAE, and the penalty barely mattered.

**Limitations.** Only 5 features, so escalation and long tickets are hard to predict. The test sets are small (180 and 160 tickets), so the test scores are a bit noisy. The model can't predict the rare very long tickets.

## Reflections

1. **Why stratify?** Only 21% of tickets escalate. Stratifying keeps that share the same in train and test, so the test score isn't skewed by chance.
2. **Why preprocessing inside the pipeline?** So the medians and scaling are learned from the training folds only. Otherwise test data leaks into training and scores look too good.
3. **What does fold variability tell us?** How much the score depends on which tickets are in the fold. If two models differ by less than the std, the difference isn't really meaningful.
4. **Why test only once?** If we keep checking the test set and changing things, we are tuning on it, and it stops being an unseen exam.
5. **Deployment constraints?** If predictions need to be fast or explainable to managers, logistic regression and Ridge are good choices. k-NN gets slow with lots of data. If missing an escalation is very costly, we'd choose a model with higher recall even if it gives more false alarms.